# RoDiLoCo on a free T4 (Kaggle / Colab)

Runs the whole reproduce → attack → defend loop on one free T4 GPU, within Kaggle's
**30 GPU-hrs/week**. Nothing here needs an A100.

**Kaggle setup:** Notebook → Settings → *Accelerator: GPU T4 x1*, *Internet: On*.
**Colab setup:** Runtime → Change runtime type → *T4 GPU*.

Rough budget: P1 <1h · P2 baseline+sweep ~2–4h · P3 grid ~3–5h · P4 campaign ~3–5h.

## 1. Get the code
Set `REPO_URL` to your pushed GitHub repo. (On Kaggle you can also add the repo as a
*Dataset* and skip the clone.)

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/<your-username>/RodliCo.git"  # <-- edit me
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.chdir(WORK)
REPO_DIR = os.path.join(WORK, "RodliCo")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print("cwd:", os.getcwd())

## 2. Dependencies
torch is preinstalled on Kaggle/Colab; we only add the extras.

In [ ]:
!pip -q install pyyaml datasets matplotlib tqdm
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 3. Sanity: unit tests (seconds)

In [ ]:
!PYTHONPATH=src python -m pytest -q

## 4. Data — TinyStories subset
~8 MB of text; plenty for a 10 M-param char model. Raise `--max-chars` for longer runs.

In [ ]:
!python scripts/prepare_data.py --out data/tinystories_train.txt --max-chars 8000000

## 5. Phase 1 — the from-scratch transformer converges
Reuses the free-tier model on the TinyStories text.

In [ ]:
import yaml
from rodiloco.train import train_single

cfg = yaml.safe_load(open("configs/free_tier.yaml"))
p1 = {**cfg, "steps": 2000, "warmup": 100, "lr": 1e-3,
      "out_dir": "outputs/p1"}
_ = train_single(p1)

## 6. Phase 2 — DiLoCo baseline (mean, no attack)
This is your "I reproduced DiLoCo" run. Sweep `H` for the comm plot.

In [ ]:
from rodiloco.diloco import run_diloco

base = yaml.safe_load(open("configs/free_tier.yaml"))
base["out_dir"] = "outputs/p2_baseline"
r_base = run_diloco(base)
print("baseline final ppl:", r_base["final_ppl"],
      "| comm reduction vs sync:", r_base["comm_reduction_vs_sync"], "x")

## 7. Phase 3 + 4 — one attack, several defenses
A minimal comparative slice you can run in one session. Expand to the full grids with
`scripts/reproduce_phase3_attacks.sh` / `reproduce_phase4_defenses.sh`.

In [ ]:
rows = []
for agg in ["mean", "trimmed_mean", "krum", "geometric_median", "centered_clipping", "trust_weighted"]:
    for f in [0, 1, 2]:
        cfg = yaml.safe_load(open("configs/free_tier.yaml"))
        cfg.update(aggregator=agg, attack="sign_flip", attack_lam=10.0, n_byzantine=f,
                   out_dir=f"outputs/p4_{agg}_f{f}")
        res = run_diloco(cfg)
        rows.append((agg, f, res["final_ppl"]))
        print(f"{agg:18s} f={f}  final_ppl={res['final_ppl']:.2f}")
print("\naggregator,f,final_ppl")
for r in rows: print(",".join(map(str, r)))

## 8. Plots — the headline figures

In [ ]:
!python scripts/plot_results.py defense 'outputs/p4_*' results/plot2_defense.png
!python scripts/plot_results.py tax 'outputs/p4_*_f0' results/plot3_tax.png
from IPython.display import Image, display
for p in ["results/plot2_defense.png", "results/plot3_tax.png"]:
    display(Image(p))

## 9. Keep the results
On Kaggle: **Save Version** persists `/kaggle/working`. Or commit `outputs/*/history.json`
and `results/*.png` back to git (they are small).

In [ ]:
import glob, json
for h in sorted(glob.glob("outputs/*/history.json")):
    d = json.load(open(h))
    print(f"{h:40s} final_ppl={d.get('final_ppl')}")